In [ ]:
import pandas as pd

In [ ]:
proportion_out = pd.read_csv(r'E:\Data\heatwave\baidu_migration\百度迁徙规模指数+迁徙OD占比数据(含迁入迁出)-所有城市-2024年度\迁徙OD占比数据\2024-01-01至2024-12-31_OD比例_迁出城市级别.csv', index_col=None)

In [ ]:
proportion_out

In [ ]:
index_out = pd.read_csv(r'E:\Data\heatwave\baidu_migration\百度迁徙规模指数+迁徙OD占比数据(含迁入迁出)-所有城市-2024年度\迁徙规模指数\2024-01-23至2024-12-31_迁出日级指数.csv', index_col=None)

In [ ]:
index_out

In [ ]:
import pandas as pd
import os

# 设置文件夹路径
folder = r'E:\Data\heatwave\baidu_migration\百度迁徙规模指数+迁徙OD占比数据(含迁入迁出)-所有城市-2024年度'

# 构造完整路径
file1 = os.path.join(folder, '2023.01.01-2023.07.16_迁出日级指数.xlsx')
file2 = os.path.join(folder, '2023.07.17-2023.12.13_迁出日级指数.xlsx')
file3 = os.path.join(folder, '2023-12-14至2023-12-31-迁出日级指数.xlsx')

# 读取 Excel 文件
df1 = pd.read_excel(file1)
df2 = pd.read_excel(file2)
df3 = pd.read_excel(file3)

# 合并数据
merged_df = pd.concat([df1, df2, df3], ignore_index=True)

merged_df = merged_df[merged_df['城市'] != '全国']

In [ ]:
merged_df['城市'].unique()

In [ ]:
merged_df[merged_df['城市']=='黔西南州']

In [ ]:
import pandas as pd

def convert_mixed_dates(date_col):
    def parse_date(val):
        if pd.isna(val):
            return pd.NaT
        if isinstance(val, (int, float)):
            val = str(int(val))
        elif isinstance(val, pd.Timestamp):
            return val
        elif isinstance(val, str) and '-' in val:
            try:
                return pd.to_datetime(val)
            except:
                return pd.NaT
        if len(val) == 8 and val.isdigit():
            try:
                return pd.to_datetime(val, format='%Y%m%d')
            except:
                return pd.NaT
        return pd.NaT

    return date_col.apply(parse_date)

# 假设你的 DataFrame 是 merged_df，日期列名为 'date'
merged_df['日期'] = convert_mixed_dates(merged_df['日期'])


In [ ]:
merged_df

In [ ]:
output_path = os.path.join(folder, '合并后的迁出日级指数.csv')
merged_df.to_csv(output_path, index=False, encoding='utf-8-sig')

In [ ]:
import pandas as pd
import numpy as np

# 1. 先把两个表的日期列都转成 datetime
index_out['日期'] = pd.to_datetime(index_out['日期'])
merged_df['日期']  = pd.to_datetime(merged_df['日期'])

# 2. 从 merged_df 里筛选出 2023-01-01 ~ 2023-01-22 这段数据
mask = (merged_df['日期'] >= '2023-01-01') & (merged_df['日期'] <= '2023-01-22')
prefix = merged_df.loc[mask].copy()

# 3. 日期 + 1 年 → 变成 2024-01-01 ~ 2024-01-22
prefix['日期'] = prefix['日期'] + pd.DateOffset(years=1)

# 4. 给它补上 index_out 里才有的“序号”列（全 NaN 或者你也可以先留空）
prefix['序号'] = np.nan

# 5. 保证列顺序一致（多余的列会自动补 NaN）
cols = index_out.columns
prefix = prefix.reindex(columns=cols)

# 6. 拼接、排序、重置索引
full = pd.concat([prefix, index_out], ignore_index=True)
full = full.sort_values(['省份','城市','类型','日期']).reset_index(drop=True)

# 7. （可选）重新生成“序号”：按 省份-城市-类型 分组，顺序编号
full['序号'] = full.groupby(['省份','城市','类型']).cumcount() + 1

# 结果就是：full 里会包含 2024-01-01 到 2024-01-22（从 2023 数据搬来并移到 2024），以及原来的 2024-01-23 开始的所有行


In [ ]:
proportion_out['比例'] = proportion_out['比例'].str.rstrip('%').astype(float)
proportion_out['比例'] = proportion_out['比例'] / 100
proportion_out['日期'] = proportion_out['日期'].astype(str)
proportion_out['日期'] = pd.to_datetime(proportion_out['日期'], format='%Y%m%d')

In [ ]:
proportion_out['日期'].dtype

In [ ]:
full.to_csv('index_2024.csv', index=False, encoding='utf-8-sig')
proportion_out.to_csv('proportion_2024.csv', index=False, encoding='utf-8-sig')

In [ ]:
replacement_dict = {
    '临夏': '临夏回族自治州',
    '伊犁': '伊犁哈萨克自治州',
    '克孜勒苏': '克孜勒苏柯尔克孜自治州',
    '临夏州': '临夏回族自治州',
    '临高': '临高县',
    '伊犁州': '伊犁哈萨克自治州',
    '克孜勒苏州': '克孜勒苏柯尔克孜自治州',
    '吉林市': '吉林',
    '大理': '大理白族自治州',
    '大理州': '大理白族自治州',
    '定安': '定安县',
    '屯昌': '屯昌县',
    '延边': '延边朝鲜族自治州',
    '德宏': '德宏傣族景颇族自治州',
    '怒江': '怒江傈僳族自治州',
    '果洛': '果洛藏族自治州',
    '楚雄': '楚雄彝族自治州',
    '海北': '海北藏族自治州',
    '海南': '海南藏族自治州',
    '海西': '海西蒙古族藏族自治州',
    '湘西': '湘西土家族苗族自治州',
    '德宏州': '德宏傣族景颇族自治州',
    '怒江州': '怒江傈僳族自治州',
    '果洛州': '果洛藏族自治州',
    '楚雄州': '楚雄彝族自治州',
    '海北州': '海北藏族自治州',
    '海南州': '海南藏族自治州',
    '海西州': '海西蒙古族藏族自治州',
    '湘西州': '湘西土家族苗族自治州',

    '澄迈': '澄迈县',
    '塔城':'塔城地区',
    '恩施':'恩施土家族苗族自治州',
    '凉山州':'凉山彝族自治州',
    
    '玉树': '玉树藏族自治州',
    '甘南': '甘南藏族自治州',
    '甘孜': '甘孜藏族自治州',
    '红河': '红河哈尼族彝族自治州',
    '迪庆': '迪庆藏族自治州',
    '阿坝': '阿坝藏族羌族自治州',
    '黄南': '黄南藏族自治州',
    '黔西南': '黔西南布依族苗族自治州',
    '昌吉':'昌吉回族自治州',
    '文山':'文山壮族苗族自治州',
    
    '玉树州': '玉树藏族自治州',
    '甘南州': '甘南藏族自治州',
    '甘孜州': '甘孜藏族自治州',
    '红河州': '红河哈尼族彝族自治州',
    '迪庆州': '迪庆藏族自治州',
    '阿坝州': '阿坝藏族羌族自治州',
    '黄南州': '黄南藏族自治州',
    '黔南州':'黔南布依族苗族自治州',
    '黔东南州': '黔东南苗族侗族自治州',
    '黔西南州': '黔西南布依族苗族自治州',
    '山南': '山南地区',
    '日喀则': '日喀则地区',
    '昌都': '昌都地区',
    '林芝': '林芝地区',
    '毕节': '毕节地区',
    '海东': '海东地区',
    '那曲': '那曲地区',
    '铜仁': '铜仁地区',
    '吐鲁番': '吐鲁番地区',
    '哈密':'哈密地区',
    '昌吉州':'昌吉回族自治州',
    '博尔塔拉州':'博尔塔拉蒙古自治州'
}

In [ ]:
proportion_out

In [ ]:
proportion_out['城市'] = proportion_out['城市'].str.replace('市$', '', regex=True)
proportion_out['目的城市'] = proportion_out['目的城市'].str.replace('市$', '', regex=True)


In [ ]:
proportion_out['城市'] = proportion_out['城市'].map(replacement_dict).fillna(proportion_out['城市'])
proportion_out['目的城市'] = proportion_out['目的城市'].map(replacement_dict).fillna(proportion_out['目的城市'])

In [ ]:
index_2024['日期'] = pd.to_datetime(
    index_2024['日期'],
    format='%Y-%m-%d',    # 如果你的字符串里没有时分秒的话
    errors='coerce'       # 解析失败的会变成 NaT，方便你排查
)

In [ ]:
proportion_2024['日期'] = pd.to_datetime(
    proportion_2024['日期'],
    format='%Y-%m-%d',    # 如果你的字符串里没有时分秒的话
    errors='coerce'       # 解析失败的会变成 NaT，方便你排查
)

In [ ]:
import pandas as pd

# 假设两个 DataFrame 分别是 proportion_2024 和 index_2024，
# 且它们的“城市”和“日期”列都已是 datetime 格式

# 1. 只保留 index_2024 中你要的“三列”──城市、日期 和 迁徙指数
tmp = index_2024[['城市', '日期', '迁徙指数']].copy()

# 2. 跟 proportion_2024 左连接
merged = pd.merge(
    proportion_2024,
    tmp,
    on=['城市', '日期'],
    how='left'
)

# —— 2. 指定要填充的缺失城市 —— #
missing_cities = [
    '黔西南布依族苗族自治州',
    '黔东南苗族侗族自治州',
    '黔南布依族苗族自治州'
]

# —— 3. 从 index_2024 中取出每个城市 2024-01-23 和 2024-12-31 的“迁徙指数”作为随机边界 —— #
bounds = {}
for city in missing_cities:
    lo = index_2024.loc[
        (index_2024['城市'] == city) & (index_2024['日期'] == pd.to_datetime('2024-01-23')),
        '迁徙指数'
    ]
    hi = index_2024.loc[
        (index_2024['城市'] == city) & (index_2024['日期'] == pd.to_datetime('2024-12-31')),
        '迁徙指数'
    ]
    if not lo.empty and not hi.empty:
        bounds[city] = (lo.iloc[0], hi.iloc[0])
    else:
        # 如果没找到，就给出提醒
        print(f"Warning: 城市 {city} 缺少边界值，无法填充！")

# —— 4. 对每个城市的缺失行，用 np.random.uniform 在边界区间里随机抽样 —— #
np.random.seed(42)  # 如果你想要可复现的“随机”结果
for city, (low, high) in bounds.items():
    mask = (merged['城市'] == city) & (merged['迁徙指数'].isna())
    cnt = mask.sum()
    if cnt:
        merged.loc[mask, '迁徙指数'] = np.random.uniform(low, high, size=cnt)

# —— 5. 验证一下还有没有剩余的 NaN —— #
print("填充后剩余缺失：", merged['迁徙指数'].isna().sum())

In [ ]:
import pandas as pd

# 假设你的 DataFrame 是 merged

# 1. 只保留需要的列
df2 = merged[['日期', '城市', '目的城市', '比例', '迁徙指数']].copy()

# 2. 确保 '日期' 列是 datetime 类型
df2['日期'] = pd.to_datetime(df2['日期'])

# 3. 用 .dt.dayofyear 计算这一年中的第几天
df2['day_index'] = df2['日期'].dt.dayofyear

# 查看结果
print(df2.head())
